# Simple bifurcation example — time-layered shortest-path revision

This notebook uses a **non-autonomous / time-layered** version of the terminal-state landscape.
Every trajectory has one state at each `time_idx`, and every terminal state is labelled `good` or `bad`.

At forecast time $t_k$, a control may switch from the current member $m$ to another member $n$ **only at the same time $t_k$**. The switching cost is

$$
d\left(x_m(t_k),x_n(t_k)\right).
$$

After that single switch, the chosen member advances naturally to $t_{k+1}$. Hence the model does **not** permit jumps to past/future states of another trajectory, nor repeated same-time switching within one forecast step.

For $p=\infty$ the recursion uses a minimax/bottleneck cost; for $p=1$ it uses an additive cumulative cost. The resulting landscape is time dependent: $C_{A,p}(t,x)$ and $S_p(t,x)$.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from sklearn.isotonic import IsotonicRegression

# Make the repository importable whether this notebook is launched from
# the repository root or from the examples directory.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "epsbasin").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import epsbasin
import epsbasin.shortest_path as shortest_path_module

print("epsbasin.shortest_path loaded from:", shortest_path_module.__file__)
assert hasattr(
    shortest_path_module,
    "minimum_epsilon_and_paths_to_good_bad",
), (
    "The loaded epsbasin/shortest_path.py is an older version. "
    "Replace it with shortest_path_external_paths_v3.py from this bundle."
)

from epsbasin.shortest_path import (
    build_terminal_state_graph,
    eps_attracting_basin_shortest_path,
    eps_sum_attracting_basin_shortest_path,
    minimum_epsilon_to_good_bad,
    minimum_epsilon_and_paths_to_good_bad,
)

## Generate the same type of two-branch time-series data

Only the last observation of each sequence is assigned `good` or `bad`. Intermediate observations are labelled `others`.

In [ ]:
def hill_function(x, hill_coef=5, hill_half=2.0):
    x = np.asarray(x, dtype=float)
    return x**hill_coef / (hill_half**hill_coef + x**hill_coef)


def uniform_disk_noise(n, scale, rng):
    radius = scale * np.sqrt(rng.uniform(size=n))
    angle = rng.uniform(0.0, 2.0 * np.pi, size=n)
    return np.column_stack([radius * np.cos(angle), radius * np.sin(angle)])


# Parameters
xlim = [0.0, 2.0]
xdiv = 20
scale = 5.0
hill_coef = 5
hill_half = 2.0
I = 20
noise_scale = 0.1
random_state = 0

rng = np.random.default_rng(random_state)
x = np.linspace(xlim[0], xlim[1], xdiv)
y = hill_function(x, hill_coef=hill_coef, hill_half=hill_half)

base_top = np.column_stack([x, y])
base_bottom = np.column_stack([x, -y])

sequences = []
seq_ids = []
time_indices = []
branch_labels = []

seq_id = 0
for branch_name, base in [("top", base_top), ("bottom", base_bottom)]:
    for _ in range(I):
        seq = base + uniform_disk_noise(xdiv, noise_scale, rng)
        sequences.append(scale * seq)
        seq_ids.extend([seq_id] * xdiv)
        time_indices.extend(range(xdiv))
        branch_labels.extend([branch_name] * xdiv)
        seq_id += 1

data_2d_flat = np.vstack(sequences)
seq_ids = np.asarray(seq_ids, dtype=int)
time_indices = np.asarray(time_indices, dtype=int)
branch_labels = np.asarray(branch_labels, dtype=object)

cluster = np.full(data_2d_flat.shape[0], "others", dtype=object)
for sid in np.unique(seq_ids):
    idx = np.flatnonzero(seq_ids == sid)
    terminal = idx[np.argmax(time_indices[idx])]
    cluster[terminal] = "good" if data_2d_flat[terminal, 1] > 0 else "bad"

adata = ad.AnnData(X=data_2d_flat)
adata.obs = pd.DataFrame(
    {
        "seq_id": seq_ids,
        "time_idx": time_indices,
        "cluster": cluster,
        "branch": branch_labels,
        "x": data_2d_flat[:, 0],
        "y": data_2d_flat[:, 1],
    },
    index=adata.obs_names.copy(),
)

adata

In [ ]:
# Confirm that every sequence terminal is labelled good or bad.
terminal_rows = adata.obs.groupby("seq_id", sort=False).tail(1)
print(terminal_rows["cluster"].value_counts())
assert terminal_rows["cluster"].isin(["good", "bad"]).all()


## Input trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for sid in adata.obs["seq_id"].unique():
    idx = np.flatnonzero(adata.obs["seq_id"].to_numpy() == sid)
    ax.plot(adata.X[idx, 0], adata.X[idx, 1], "-", lw=1, alpha=0.35)

mask_good = adata.obs["cluster"].to_numpy() == "good"
mask_bad = adata.obs["cluster"].to_numpy() == "bad"
ax.scatter(adata.X[mask_good, 0], adata.X[mask_good, 1], s=60, label="good", zorder=5)
ax.scatter(adata.X[mask_bad, 0], adata.X[mask_bad, 1], s=60, label="bad", zorder=5)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Input sequences and terminal states")
ax.grid(ls="--", alpha=0.4)
ax.legend()
plt.show()


## Velocity vectors

The velocity is computed independently within each sequence by central differences, with one-sided differences at the endpoints.

In [ ]:
velocity = np.zeros_like(adata.X, dtype=float)

for sid in adata.obs["seq_id"].unique():
    idx = np.flatnonzero(adata.obs["seq_id"].to_numpy() == sid)
    points = np.asarray(adata.X[idx], dtype=float)

    if len(idx) == 1:
        velocity[idx] = 0.0
    elif len(idx) == 2:
        v = points[1] - points[0]
        velocity[idx[0]] = v
        velocity[idx[1]] = v
    else:
        velocity[idx[0]] = points[1] - points[0]
        velocity[idx[-1]] = points[-1] - points[-2]
        velocity[idx[1:-1]] = 0.5 * (points[2:] - points[:-2])

adata.obsm["velocity"] = velocity

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(adata.X[:, 0], adata.X[:, 1], s=8, alpha=0.35)
ax.quiver(
    adata.X[:, 0],
    adata.X[:, 1],
    velocity[:, 0],
    velocity[:, 1],
    angles="xy",
    scale_units="xy",
    scale=1.0,
    width=0.002,
    alpha=0.5,
)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Velocity field")
ax.grid(ls="--", alpha=0.4)
plt.show()


## Build and inspect the time-layered transition graph

With `transition_mode="time_layered"`, **cross-sequence costs are finite only between states with the same `time_idx`**. Same-sequence one-step forward dynamics has cost zero, while other inter-time jumps are infinite.

The inspection graph therefore implements the literal time restriction. The actual Good/Bad reach costs are computed by backward dynamic programming rather than by Dijkstra on this graph, so that at most one same-time member switch is allowed per forecast step.


In [ ]:
graph = build_terminal_state_graph(
    adata,
    seq_key="seq_id",
    time_key="time_idx",
    cluster_key="cluster",
    good_cluster_key="good",
    bad_cluster_key="bad",
    sequence_edge_mode="adjacent",
    transition_mode="time_layered",
)

# Sanity checks for the literal time restriction.
seq_values = adata.obs["seq_id"].to_numpy(dtype=int)
time_values = adata.obs["time_idx"].to_numpy(dtype=int)
first_seq = int(np.unique(seq_values)[0])
other_seq = int(np.unique(seq_values)[-1])

idx0 = np.flatnonzero(seq_values == first_seq)
idx0 = idx0[np.argsort(time_values[idx0])]
idx1 = np.flatnonzero(seq_values == other_seq)
idx1 = idx1[np.argsort(time_values[idx1])]

print("natural advance t0 -> t1:", graph.weights[idx0[0], idx0[1]])
print("forbidden direct t0 -> t2:", graph.weights[idx0[0], idx0[2]])
print("same-time cross-member t0 -> t0:", graph.weights[idx0[0], idx1[0]])
print("forbidden cross-member t0 -> t1:", graph.weights[idx0[0], idx1[1]])
print("number of terminal states:", len(graph.terminal_indices))


## New $\varepsilon$ debut functions: time-layered minimax recursion

For $p=\infty$, the non-autonomous recursion is

$$
C_{A,\infty}(m,k)
=
\min_n
\max\left\{
 d\bigl(x_m^k,x_n^k\bigr),
 C_{A,\infty}(n,k+1)
\right\}.
$$

Thus a trajectory can switch member at the current time but cannot jump to a different time index.

In [ ]:
eps_attracting_basin_shortest_path(
    adata,
    seq_key="seq_id",
    time_key="time_idx",
    cluster_key="cluster",
    good_cluster_key="good",
    bad_cluster_key="bad",
    cost_matrix_key="cost_matrix",
    sequence_edge_mode="adjacent",
    transition_mode="time_layered",
    output_key="eps_attracting_basin_sp",
    distance_key="eps_distance_sp",
    terminal_class_key="terminal_class",
    store_next_hop=True,
)

adata.obs[[
    "cluster",
    "time_idx",
    "terminal_class",
    "eps_attracting_basin_sp_good",
    "eps_attracting_basin_sp_bad",
    "eps_attracting_basin_sp_landscape",
]].head()

## New $\varepsilon_\Sigma$ debut functions: time-layered additive recursion

For $p=1$, the cumulative-cost recursion is

$$
C_{A,1}(m,k)
=
\min_n
\left[
 d\bigl(x_m^k,x_n^k\bigr)
 + C_{A,1}(n,k+1)
\right].
$$

Only one same-time member switch is allowed at each forecast step.

In [ ]:
eps_sum_attracting_basin_shortest_path(
    adata,
    seq_key="seq_id",
    time_key="time_idx",
    cluster_key="cluster",
    good_cluster_key="good",
    bad_cluster_key="bad",
    cost_matrix_key="cost_matrix",
    sequence_edge_mode="adjacent",
    transition_mode="time_layered",
    output_key="eps_sum_attracting_basin_sp",
    distance_key="eps_sum_distance_sp",
    store_next_hop=True,
)

adata.obs[[
    "cluster",
    "time_idx",
    "eps_sum_attracting_basin_sp_good",
    "eps_sum_attracting_basin_sp_bad",
    "eps_sum_attracting_basin_sp_landscape",
]].head()

## Query an arbitrary state $x$: $C_{G,p}(x)$, $C_{B,p}(x)$, and realizing paths

After the ensemble debut functions have been learned, an external state $x$ is
queried by supplying the cost from $x$ to the ensemble states.  In the
**time-layered** model, only ensemble states at the same `time_idx` are
admissible entry states.

For each target outcome $A\in\{G,B\}$, the helper
`epsbasin.shortest_path.minimum_epsilon_and_paths_to_good_bad` returns both

$$
C_{A,p}(t_k,x)
$$

and one optimal path that realizes this value.  A path consists of

- a dashed **entry/switch** segment, which carries the perturbation cost; and
- a solid **natural-advance** segment, which follows the selected ensemble
  member to the next time layer at zero cost.

For $p=1$, the path value is the sum of the perturbation costs.  For
$p=\infty$, it is their maximum.

This notebook requires the replacement `epsbasin/shortest_path.py` supplied as `shortest_path_external_paths_v3.py`.


In [ ]:
# A query chosen early enough that the L1 and L-infinity optimal paths
# can differ.  This state is not one of the ensemble observations.
QUERY_TIME_IDX = 8
x_query = np.array([4.2, 0.2], dtype=float)

# Cost from x to the ensemble.  For the time-layered query we make the
# same-time restriction explicit by setting all other time layers to infinity.
query_costs = np.full(adata.n_obs, np.inf, dtype=float)
same_time_mask = (
    adata.obs["time_idx"].to_numpy(dtype=int) == QUERY_TIME_IDX
)
X_same_time = np.asarray(adata.X[same_time_mask], dtype=float)
query_costs[same_time_mask] = np.linalg.norm(
    X_same_time - x_query[None, :],
    axis=1,
)

query_result_inf = minimum_epsilon_and_paths_to_good_bad(
    adata,
    x_query,
    query_costs,
    output_key="eps_attracting_basin_sp",
    time_value=QUERY_TIME_IDX,
)
query_result_one = minimum_epsilon_and_paths_to_good_bad(
    adata,
    x_query,
    query_costs,
    output_key="eps_sum_attracting_basin_sp",
    time_value=QUERY_TIME_IDX,
)

for name, result in [
    (r"$p=\infty$", query_result_inf),
    (r"$p=1$", query_result_one),
]:
    print(
        f"{name}: "
        f"C_G={result.epsilon_good:.4f}, "
        f"C_B={result.epsilon_bad:.4f}, "
        f"S=C_B-C_G={result.score_good:.4f}"
    )
    print(
        "  realized by paths: "
        f"Good={result.path_good.realized_value:.4f}, "
        f"Bad={result.path_bad.realized_value:.4f}"
    )
    assert np.isclose(
        result.path_good.realized_value,
        result.epsilon_good,
    )
    assert np.isclose(
        result.path_bad.realized_value,
        result.epsilon_bad,
    )


def optimal_path_table(path, adata_obj):
    """Human-readable table of the transitions in one reconstructed path."""
    rows = []
    for step_no, step in enumerate(path.steps):
        if step.source_index is None:
            source = "x"
            source_seq = "external"
            source_time = QUERY_TIME_IDX
        else:
            source = int(step.source_index)
            source_seq = adata_obj.obs.iloc[step.source_index]["seq_id"]
            source_time = adata_obj.obs.iloc[step.source_index]["time_idx"]

        destination = int(step.destination_index)
        rows.append(
            {
                "step": step_no,
                "kind": step.kind,
                "source": source,
                "source_seq": source_seq,
                "source_time": source_time,
                "destination": destination,
                "destination_seq": adata_obj.obs.iloc[destination]["seq_id"],
                "destination_time": adata_obj.obs.iloc[destination]["time_idx"],
                "cost": step.cost,
            }
        )
    return pd.DataFrame(rows)


print("\nL-infinity path to Good")
display(optimal_path_table(query_result_inf.path_good, adata))
print("L-infinity path to Bad")
display(optimal_path_table(query_result_inf.path_bad, adata))
print("L1 path to Good")
display(optimal_path_table(query_result_one.path_good, adata))
print("L1 path to Bad")
display(optimal_path_table(query_result_one.path_bad, adata))

### State-space visualization of the optimal Good/Bad paths

The gray curves are the training ensemble trajectories.  The black star is the
external query $x$.  Red and blue show one path realizing $C_{G,p}(x)$ and
$C_{B,p}(x)$, respectively.  Dashed arrows are perturbative entry/switch
transitions; solid arrows are zero-cost natural evolution.

In [ ]:
from matplotlib.lines import Line2D


def plot_query_paths(ax, result, *, title):
    X_all = np.asarray(adata.X, dtype=float)
    seq_values = adata.obs["seq_id"].to_numpy(dtype=int)
    time_values = adata.obs["time_idx"].to_numpy(dtype=int)

    # Background ensemble trajectories.
    for seq_id in np.unique(seq_values):
        idx = np.flatnonzero(seq_values == seq_id)
        idx = idx[np.argsort(time_values[idx])]
        ax.plot(
            X_all[idx, 0],
            X_all[idx, 1],
            color="0.75",
            linewidth=0.7,
            alpha=0.45,
            zorder=1,
        )

    # Ensemble states available at the query time.
    same_time_indices = np.flatnonzero(time_values == QUERY_TIME_IDX)
    ax.scatter(
        X_all[same_time_indices, 0],
        X_all[same_time_indices, 1],
        s=22,
        facecolors="none",
        edgecolors="0.45",
        linewidths=0.6,
        zorder=2,
    )

    # Terminal outcomes.
    terminal_good = np.flatnonzero(
        adata.obs["cluster"].astype(str).to_numpy() == "good"
    )
    terminal_bad = np.flatnonzero(
        adata.obs["cluster"].astype(str).to_numpy() == "bad"
    )
    ax.scatter(
        X_all[terminal_good, 0],
        X_all[terminal_good, 1],
        s=28,
        color="tab:red",
        marker="o",
        edgecolor="black",
        linewidth=0.4,
        zorder=3,
    )
    ax.scatter(
        X_all[terminal_bad, 0],
        X_all[terminal_bad, 1],
        s=28,
        color="tab:blue",
        marker="o",
        edgecolor="black",
        linewidth=0.4,
        zorder=3,
    )

    # External query state.
    ax.scatter(
        [x_query[0]],
        [x_query[1]],
        s=140,
        marker="*",
        color="black",
        edgecolor="white",
        linewidth=0.8,
        zorder=10,
    )

    def draw_optimal_path(path, color):
        visited = []
        for step in path.steps:
            if step.source_index is None:
                p0 = x_query
            else:
                p0 = X_all[int(step.source_index)]
            p1 = X_all[int(step.destination_index)]

            is_control = step.kind in {"entry", "switch", "graph"}
            linestyle = "--" if is_control else "-"
            linewidth = 2.2 if is_control else 2.8

            ax.annotate(
                "",
                xy=(p1[0], p1[1]),
                xytext=(p0[0], p0[1]),
                arrowprops=dict(
                    arrowstyle="->",
                    color=color,
                    linewidth=linewidth,
                    linestyle=linestyle,
                    shrinkA=2,
                    shrinkB=2,
                    alpha=0.95,
                ),
                zorder=7,
            )

            # Label only nonzero control costs to avoid clutter.
            if is_control and step.cost > 1e-10:
                midpoint = 0.5 * (p0 + p1)
                ax.text(
                    midpoint[0],
                    midpoint[1],
                    f"{step.cost:.2f}",
                    color=color,
                    fontsize=7,
                    ha="center",
                    va="bottom",
                    zorder=9,
                )
            visited.append(int(step.destination_index))

        if visited:
            unique_visited = np.unique(visited)
            ax.scatter(
                X_all[unique_visited, 0],
                X_all[unique_visited, 1],
                s=32,
                facecolor=color,
                edgecolor="white",
                linewidth=0.5,
                zorder=8,
            )

    draw_optimal_path(result.path_good, "tab:red")
    draw_optimal_path(result.path_bad, "tab:blue")

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    ax.set_title(
        title
        + "\n"
        + rf"$C_G={result.epsilon_good:.3f}$, "
          rf"$C_B={result.epsilon_bad:.3f}$, "
          rf"$S={result.score_good:.3f}$"
    )

    legend_handles = [
        Line2D([0], [0], color="tab:red", lw=2.5, label="optimal path to Good"),
        Line2D([0], [0], color="tab:blue", lw=2.5, label="optimal path to Bad"),
        Line2D([0], [0], color="black", lw=2.0, ls="--", label="entry / same-time switch"),
        Line2D([0], [0], color="black", lw=2.0, ls="-", label="natural advance (cost 0)"),
        Line2D([0], [0], marker="*", color="black", lw=0, markersize=10, label="query x"),
    ]
    ax.legend(
        handles=legend_handles,
        frameon=True,
        fontsize=8,
        loc="upper left",
    )


fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.6),
    constrained_layout=True,
)
plot_query_paths(
    axes[0],
    query_result_inf,
    title=rf"$p=\infty$, time_idx={QUERY_TIME_IDX}",
)
plot_query_paths(
    axes[1],
    query_result_one,
    title=rf"$p=1$, time_idx={QUERY_TIME_IDX}",
)
fig.suptitle(
    r"Optimal paths realizing $C_{G,p}(x)$ and $C_{B,p}(x)$",
    y=1.02,
)
plt.show()

## Debut functions for $\varepsilon$

In [ ]:
epsbasin.plot_debut(
    adata,
    eps_key="eps_attracting_basin_sp",
    target_cluster_key="good",
    title="Shortest-path debut: Good",
)

epsbasin.plot_debut(
    adata,
    eps_key="eps_attracting_basin_sp",
    target_cluster_key="bad",
    title="Shortest-path debut: Bad",
)


## Debut functions for $\varepsilon_\Sigma$

In [ ]:
epsbasin.plot_debut(
    adata,
    eps_key="eps_sum_attracting_basin_sp",
    target_cluster_key="good",
    title=r"Shortest-path debut ($\varepsilon_\Sigma$): Good",
)

epsbasin.plot_debut(
    adata,
    eps_key="eps_sum_attracting_basin_sp",
    target_cluster_key="bad",
    title=r"Shortest-path debut ($\varepsilon_\Sigma$): Bad",
)


## Landscape

In [ ]:
epsbasin.plot_landscape(
    adata,
    eps_key="eps_attracting_basin_sp",
    target_cluster_keys=["good", "bad"],
    figsize=(10, 10),
    show_title=True,
    title="Shortest-path landscape",
)


In [ ]:
epsbasin.plot_landscape(
    adata,
    eps_key="eps_sum_attracting_basin_sp",
    target_cluster_keys=["good", "bad"],
    figsize=(10, 10),
    show_title=True,
    title=r"Shortest-path landscape ($\varepsilon_\Sigma$)",
)


## Attracting-basin visualization at a selected threshold

The old plotting function can be reused because the new implementation writes Good/Bad debut functions using the same `<eps_key>_good` / `<eps_key>_bad` naming convention.

In [ ]:
eps_threshold = 0.5

epsbasin.plot_attracting_basin(
    adata,
    eps_key="eps_attracting_basin_sp",
    cluster_key="cluster",
    good_cluster_key="good",
    bad_cluster_key="bad",
    eps_threshold=eps_threshold,
    figsize=(8, 8),
    title=rf"Shortest-path $\varepsilon$-attracting basin ($\varepsilon={eps_threshold}$)",
)


## Basic consistency checks

In [ ]:
# Each observation inherits the natural terminal class of its own sequence.
for sid in adata.obs["seq_id"].unique():
    idx = np.flatnonzero(adata.obs["seq_id"].to_numpy() == sid)
    natural_terminal = adata.obs["cluster"].iloc[idx[-1]]
    assert np.all(adata.obs["terminal_class"].iloc[idx].to_numpy() == natural_terminal)

# The two signed debut functions are complementary under the Good/Bad terminal partition.
np.testing.assert_allclose(
    adata.obs["eps_attracting_basin_sp_good"].to_numpy(),
    -adata.obs["eps_attracting_basin_sp_bad"].to_numpy(),
)
np.testing.assert_allclose(
    adata.obs["eps_sum_attracting_basin_sp_good"].to_numpy(),
    -adata.obs["eps_sum_attracting_basin_sp_bad"].to_numpy(),
)

print("All consistency checks passed.")


## Leave-one-sequence-out cross-validation

For each `seq_id`, the complete trajectory is removed before constructing the landscape. For a held-out state $x$ at time $t_k$, **only training states at the same `time_idx` are eligible entry states**.

For a non-terminal time,

$$
C_{A,\infty}(t_k,x)
=
\min_n
\max\left\{
 d\bigl(x,x_n(t_k)\bigr),
 C_{A,\infty}\bigl(n,t_{k+1}\bigr)
\right\},
$$

and

$$
C_{A,1}(t_k,x)
=
\min_n
\left[
 d\bigl(x,x_n(t_k)\bigr)
 + C_{A,1}\bigl(n,t_{k+1}\bigr)
\right].
$$

The terminal label of the held-out trajectory is used only for scoring the prediction, never for training.

In [ ]:
def evaluate_external_state(
    adata_train,
    x,
    costs,
    *,
    output_key,
    time_value,
):
    """Evaluate a held-out/background state using the time-layered package helper."""
    return minimum_epsilon_to_good_bad(
        adata_train,
        x,
        costs,
        output_key=output_key,
        time_value=time_value,
    )

In [ ]:
def leave_one_sequence_out_cv(
    adata_full,
    *,
    path_cost="max",
    seq_key="seq_id",
    time_key="time_idx",
    cluster_key="cluster",
    good_cluster_key="good",
    bad_cluster_key="bad",
    cost_matrix_key="cost_matrix",
    tie_tol=1e-12,
    verbose=True,
):
    """LOSO-CV with strict same-time entry and time-layered dynamics."""
    if path_cost not in {"max", "sum"}:
        raise ValueError("path_cost must be 'max' or 'sum'.")

    seq_values = adata_full.obs[seq_key].to_numpy()
    time_values = adata_full.obs[time_key].to_numpy()
    all_indices = np.arange(adata_full.n_obs)
    seq_ids_unique = list(pd.unique(seq_values))

    if cost_matrix_key in adata_full.uns:
        full_cost = np.asarray(adata_full.uns[cost_matrix_key], dtype=float)
    else:
        from sklearn.metrics import pairwise_distances
        full_cost = pairwise_distances(adata_full.X)

    if full_cost.shape != (adata_full.n_obs, adata_full.n_obs):
        raise ValueError("The full cost matrix has an incompatible shape.")

    rows = []

    for fold, heldout_seq in enumerate(seq_ids_unique, start=1):
        test_indices = all_indices[seq_values == heldout_seq]
        test_indices = test_indices[np.argsort(time_values[test_indices])]
        train_indices = all_indices[seq_values != heldout_seq]

        adata_train = adata_full[train_indices].copy()
        adata_train.uns[cost_matrix_key] = full_cost[
            np.ix_(train_indices, train_indices)
        ].copy()

        if path_cost == "max":
            output_key = "cv_eps_attracting_basin_sp"
            eps_attracting_basin_shortest_path(
                adata_train,
                seq_key=seq_key,
                time_key=time_key,
                cluster_key=cluster_key,
                good_cluster_key=good_cluster_key,
                bad_cluster_key=bad_cluster_key,
                cost_matrix_key=cost_matrix_key,
                sequence_edge_mode="adjacent",
                transition_mode="time_layered",
                output_key=output_key,
            )
        else:
            output_key = "cv_eps_sum_attracting_basin_sp"
            eps_sum_attracting_basin_shortest_path(
                adata_train,
                seq_key=seq_key,
                time_key=time_key,
                cluster_key=cluster_key,
                good_cluster_key=good_cluster_key,
                bad_cluster_key=bad_cluster_key,
                cost_matrix_key=cost_matrix_key,
                sequence_edge_mode="adjacent",
                transition_mode="time_layered",
                output_key=output_key,
            )

        true_label = str(adata_full.obs.iloc[test_indices[-1]][cluster_key])
        if true_label not in {good_cluster_key, bad_cluster_key}:
            raise ValueError(
                f"Held-out sequence {heldout_seq!r} does not terminate in "
                f"{good_cluster_key!r} or {bad_cluster_key!r}."
            )

        train_time_values = adata_train.obs[time_key].to_numpy()

        for step, global_index in enumerate(test_indices):
            time_value = time_values[global_index]
            x_query = np.asarray(
                adata_full.X[global_index],
                dtype=float,
            ).reshape(-1)

            # Same-time entry only. The package helper also enforces this, but
            # setting the other entries to inf makes the rule explicit here.
            costs_x_to_train = np.asarray(
                full_cost[global_index, train_indices],
                dtype=float,
            ).copy()
            costs_x_to_train[train_time_values != time_value] = np.inf

            result = evaluate_external_state(
                adata_train,
                x_query,
                costs_x_to_train,
                output_key=output_key,
                time_value=time_value,
            )

            score_good = result.epsilon_bad - result.epsilon_good
            is_tie = bool(abs(score_good) <= tie_tol)
            # Fixed landscape decision rule used throughout this notebook:
            # S_p > 0 -> Good, S_p <= 0 -> Bad. Exact ties therefore go to Bad.
            prediction = (
                good_cluster_key
                if score_good > 0.0
                else bad_cluster_key
            )

            entry_good_local = result.entry_index_good
            entry_bad_local = result.entry_index_bad
            entry_good_global = (
                -1
                if entry_good_local < 0
                else int(train_indices[entry_good_local])
            )
            entry_bad_global = (
                -1
                if entry_bad_local < 0
                else int(train_indices[entry_bad_local])
            )

            rows.append(
                {
                    "path_cost": path_cost,
                    "seq_id": heldout_seq,
                    "step": step,
                    "time_idx": time_value,
                    "fraction": (
                        0.0
                        if len(test_indices) == 1
                        else step / (len(test_indices) - 1)
                    ),
                    "global_index": int(global_index),
                    "x": float(adata_full.X[global_index, 0]),
                    "y": float(adata_full.X[global_index, 1]),
                    "true_label": true_label,
                    "epsilon_good": result.epsilon_good,
                    "epsilon_bad": result.epsilon_bad,
                    "margin_good": score_good,
                    "prediction": prediction,
                    "is_tie": is_tie,
                    "correct": prediction == true_label,
                    "entry_good_global_index": entry_good_global,
                    "entry_bad_global_index": entry_bad_global,
                }
            )

        if verbose:
            print(
                f"[{fold:02d}/{len(seq_ids_unique):02d}] "
                f"held out seq_id={heldout_seq!r}"
            )

    return pd.DataFrame(rows)

### Run LOSO-CV for ordinary $\varepsilon$

Each training landscape is recomputed with `transition_mode="time_layered"`. The held-out query can enter only the training states at its own `time_idx`.

In [ ]:
cv_eps = leave_one_sequence_out_cv(
    adata,
    path_cost="max",
    seq_key="seq_id",
    time_key="time_idx",
    cluster_key="cluster",
    good_cluster_key="good",
    bad_cluster_key="bad",
    cost_matrix_key="cost_matrix",
    verbose=True,
)

cv_eps.head(10)

### Terminal-state prediction accuracy

The last state of each held-out trajectory is one independent terminal
prediction.  `accuracy_all` counts ties as incorrect; `accuracy_resolved`
reports accuracy only among non-tied predictions.

In [ ]:
terminal_cv = (
    cv_eps.sort_values(["seq_id", "step"])
    .groupby("seq_id", sort=False)
    .tail(1)
    .reset_index(drop=True)
)

accuracy_all = terminal_cv["correct"].mean()
tie_rate = terminal_cv["is_tie"].mean()

print(f"terminal accuracy (S>0 Good, else Bad): {accuracy_all:.3f}")
print(f"terminal exact-tie rate:                {tie_rate:.3f}")

display(terminal_cv[[
    "seq_id",
    "true_label",
    "epsilon_good",
    "epsilon_bad",
    "margin_good",
    "prediction",
    "is_tie",
    "correct",
]])


### Time-dependent score surfaces $S_p(t,x)=C_{B,p}(t,x)-C_{G,p}(t,x)$

Because the system is treated as non-autonomous, the landscape is now **time dependent**. Choose a forecast layer with `SURFACE_TIME_IDX` below. Only ensemble states at that same time are admissible switching targets.

For a non-terminal layer,

$$
S_p(t_k,x)=C_{B,p}(t_k,x)-C_{G,p}(t_k,x),
$$

with $S_p>0$ favoring Good. The default is the second-to-last time layer, where the two outcomes are already well separated but the terminal outcome has not yet been reached.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

SURFACE_TIME_IDX = int(adata.obs["time_idx"].max()) - 1
SURFACE_NX = 101
SURFACE_NY = 101
SAVE_SCORE_SURFACE_FIGURES = False
SCORE_SURFACE_OUTPUT_DIR = Path("figures")
if SAVE_SCORE_SURFACE_FIGURES:
    SCORE_SURFACE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _finite_symmetric_limit(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return 1.0
    value = float(np.max(np.abs(finite)))
    return 1.0 if value == 0.0 else value


def _time_layer_indices(adata_obj, time_value):
    seq_values = adata_obj.obs["seq_id"].to_numpy()
    time_values = adata_obj.obs["time_idx"].to_numpy()
    seq_ids_here = list(pd.unique(seq_values))
    unique_times = np.sort(pd.unique(time_values))

    matches = np.flatnonzero(unique_times == time_value)
    if matches.size != 1:
        raise ValueError(f"time_idx={time_value!r} is not in the data.")
    t_pos = int(matches[0])

    current = []
    following = []
    for sid in seq_ids_here:
        idx = np.flatnonzero(seq_values == sid)
        idx = idx[np.argsort(time_values[idx])]
        current_match = idx[time_values[idx] == time_value]
        if current_match.size != 1:
            raise ValueError(
                f"seq_id={sid!r} has {current_match.size} states at time {time_value}."
            )
        current.append(int(current_match[0]))
        if t_pos < unique_times.size - 1:
            next_time = unique_times[t_pos + 1]
            next_match = idx[time_values[idx] == next_time]
            if next_match.size != 1:
                raise ValueError(
                    f"seq_id={sid!r} has {next_match.size} states at time {next_time}."
                )
            following.append(int(next_match[0]))

    return (
        np.asarray(current, dtype=int),
        None if not following else np.asarray(following, dtype=int),
        unique_times,
    )


def compute_time_layered_score_surface(
    adata_obj,
    *,
    time_value,
    distance_good_key,
    distance_bad_key,
    path_cost,
    score_label,
    nx=SURFACE_NX,
    ny=SURFACE_NY,
    pad_x=0.5,
    pad_y=0.5,
):
    """Evaluate S_p(t,x) using only the selected time layer."""
    if path_cost not in {"max", "sum"}:
        raise ValueError("path_cost must be 'max' or 'sum'.")

    X = np.asarray(adata_obj.X, dtype=float)
    current, following, unique_times = _time_layer_indices(
        adata_obj,
        time_value,
    )
    X_current = X[current]

    x_grid = np.linspace(X[:, 0].min() - pad_x, X[:, 0].max() + pad_x, nx)
    y_grid = np.linspace(X[:, 1].min() - pad_y, X[:, 1].max() + pad_y, ny)
    XX, YY = np.meshgrid(x_grid, y_grid)
    points = np.column_stack([XX.ravel(), YY.ravel()])
    entry = np.linalg.norm(
        points[:, None, :] - X_current[None, :, :],
        axis=2,
    )

    if following is None:
        labels = adata_obj.obs.iloc[current]["cluster"].astype(str).to_numpy()
        good_cols = np.flatnonzero(labels == "good")
        bad_cols = np.flatnonzero(labels == "bad")
        cost_good = np.min(entry[:, good_cols], axis=1)
        cost_bad = np.min(entry[:, bad_cols], axis=1)
    else:
        future_good = adata_obj.obs.iloc[following][
            distance_good_key
        ].to_numpy(dtype=float)
        future_bad = adata_obj.obs.iloc[following][
            distance_bad_key
        ].to_numpy(dtype=float)

        if path_cost == "max":
            cost_good = np.min(
                np.maximum(entry, future_good[None, :]),
                axis=1,
            )
            cost_bad = np.min(
                np.maximum(entry, future_bad[None, :]),
                axis=1,
            )
        else:
            cost_good = np.min(entry + future_good[None, :], axis=1)
            cost_bad = np.min(entry + future_bad[None, :], axis=1)

    score = cost_bad - cost_good
    return {
        "label": score_label,
        "time_value": time_value,
        "X": XX,
        "Y": YY,
        "score": score.reshape(YY.shape),
        "cost_good": cost_good.reshape(YY.shape),
        "cost_bad": cost_bad.reshape(YY.shape),
        "current_indices": current,
        "xlim": (x_grid[0], x_grid[-1]),
        "ylim": (y_grid[0], y_grid[-1]),
    }


def validate_time_layered_recurrence(
    adata_obj,
    *,
    distance_good_key,
    distance_bad_key,
    path_cost,
    atol=1e-10,
):
    """Check every time layer against the time-layered Bellman recursion."""
    X = np.asarray(adata_obj.X, dtype=float)
    full_cost = np.asarray(adata_obj.uns["cost_matrix"], dtype=float)
    times = np.sort(pd.unique(adata_obj.obs["time_idx"]))
    d_good = adata_obj.obs[distance_good_key].to_numpy(dtype=float)
    d_bad = adata_obj.obs[distance_bad_key].to_numpy(dtype=float)

    max_error = 0.0
    for t_pos, time_value in enumerate(times):
        current, following, _ = _time_layer_indices(adata_obj, time_value)
        switch = full_cost[np.ix_(current, current)]

        if following is None:
            labels = adata_obj.obs.iloc[current]["cluster"].astype(str).to_numpy()
            good_targets = current[labels == "good"]
            bad_targets = current[labels == "bad"]
            expected_good = np.min(
                full_cost[np.ix_(current, good_targets)],
                axis=1,
            )
            expected_bad = np.min(
                full_cost[np.ix_(current, bad_targets)],
                axis=1,
            )
        else:
            if path_cost == "max":
                expected_good = np.min(
                    np.maximum(switch, d_good[following][None, :]),
                    axis=1,
                )
                expected_bad = np.min(
                    np.maximum(switch, d_bad[following][None, :]),
                    axis=1,
                )
            else:
                expected_good = np.min(
                    switch + d_good[following][None, :],
                    axis=1,
                )
                expected_bad = np.min(
                    switch + d_bad[following][None, :],
                    axis=1,
                )

        max_error = max(
            max_error,
            float(np.max(np.abs(expected_good - d_good[current]))),
            float(np.max(np.abs(expected_bad - d_bad[current]))),
        )

    print(f"{path_cost}: time-layered recurrence max error = {max_error:.3e}")
    if max_error > atol:
        raise AssertionError(
            f"{path_cost} distances do not satisfy the time-layered recurrence."
        )


def validate_background_queries(
    adata_obj,
    *,
    time_value,
    output_key,
    surface,
    n_checks=12,
    random_state=0,
    atol=1e-10,
):
    """Compare direct surface evaluation with the package external-state helper."""
    rng = np.random.default_rng(random_state)
    current = surface["current_indices"]
    X_current = np.asarray(adata_obj.X[current], dtype=float)

    xmin, xmax = surface["xlim"]
    ymin, ymax = surface["ylim"]
    points = np.column_stack([
        rng.uniform(xmin, xmax, n_checks),
        rng.uniform(ymin, ymax, n_checks),
    ])

    for point in points:
        costs = np.full(adata_obj.n_obs, np.inf, dtype=float)
        costs[current] = np.linalg.norm(
            X_current - point[None, :],
            axis=1,
        )
        result = minimum_epsilon_to_good_bad(
            adata_obj,
            point,
            costs,
            output_key=output_key,
            time_value=time_value,
        )

        # Recompute directly at the point using the same selected layer.
        # This keeps the validation independent of grid interpolation.
        following = _time_layer_indices(adata_obj, time_value)[1]
        entry = costs[current]
        if following is None:
            labels = adata_obj.obs.iloc[current]["cluster"].astype(str).to_numpy()
            direct_good = np.min(entry[labels == "good"])
            direct_bad = np.min(entry[labels == "bad"])
        else:
            params = adata_obj.uns[f"{output_key}_params"]
            path_cost = params["path_cost"]
            d_good = np.maximum(
                adata_obj.obs[f"{output_key}_good"].to_numpy(dtype=float),
                0.0,
            )[following]
            d_bad = np.maximum(
                adata_obj.obs[f"{output_key}_bad"].to_numpy(dtype=float),
                0.0,
            )[following]
            if path_cost == "max":
                direct_good = np.min(np.maximum(entry, d_good))
                direct_bad = np.min(np.maximum(entry, d_bad))
            else:
                direct_good = np.min(entry + d_good)
                direct_bad = np.min(entry + d_bad)

        if not (
            np.isclose(direct_good, result.epsilon_good, atol=atol, rtol=0.0)
            and np.isclose(direct_bad, result.epsilon_bad, atol=atol, rtol=0.0)
        ):
            raise AssertionError("Background query does not match package helper.")

    print(f"{output_key}: {n_checks} background-query checks passed at time {time_value}.")


def _outcome_for_current_states(adata_obj, indices):
    terminal_class = adata_obj.obs.iloc[indices]["terminal_class"].astype(str).to_numpy()
    return terminal_class


def plot_score_map(ax, surface, *, vabs, title, show_legend=False):
    Xg, Yg, Zg = surface["X"], surface["Y"], surface["score"]
    levels = np.linspace(-vabs, vabs, 25)
    image = ax.contourf(
        Xg,
        Yg,
        Zg,
        levels=levels,
        cmap="coolwarm",
        vmin=-vabs,
        vmax=vabs,
        extend="both",
    )
    ax.contour(
        Xg,
        Yg,
        Zg,
        levels=[0.0],
        colors="black",
        linewidths=2.2,
    )

    current = surface["current_indices"]
    outcomes = _outcome_for_current_states(adata, current)
    for label, marker in [("good", "o"), ("bad", "^")]:
        pts = np.asarray(adata.X[current[outcomes == label]], dtype=float)
        ax.scatter(
            pts[:, 0],
            pts[:, 1],
            s=32,
            marker=marker,
            edgecolor="black",
            linewidth=0.4,
            label=f"{label}-outcome member",
            zorder=5,
        )

    ax.set_xlim(*surface["xlim"])
    ax.set_ylim(*surface["ylim"])
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    if show_legend:
        ax.legend(frameon=True, fontsize=9, loc="upper left")
    return image


def plot_score_surface_3d(ax, surface, *, vabs, zlabel):
    Xg, Yg, Zg = surface["X"], surface["Y"], surface["score"]
    surf = ax.plot_surface(
        Xg,
        Yg,
        Zg,
        cmap="coolwarm",
        vmin=-vabs,
        vmax=vabs,
        linewidth=0,
        antialiased=True,
        alpha=0.92,
    )
    ax.contour(
        Xg,
        Yg,
        Zg,
        levels=[0.0],
        colors="black",
        linewidths=2.0,
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel(zlabel)
    ax.set_zlim(-vabs, vabs)
    ax.view_init(elev=28, azim=-62)
    return surf


# Validate the dynamic programming values first.
validate_time_layered_recurrence(
    adata,
    distance_good_key="eps_distance_sp_good",
    distance_bad_key="eps_distance_sp_bad",
    path_cost="max",
)
validate_time_layered_recurrence(
    adata,
    distance_good_key="eps_sum_distance_sp_good",
    distance_bad_key="eps_sum_distance_sp_bad",
    path_cost="sum",
)

surface_inf = compute_time_layered_score_surface(
    adata,
    time_value=SURFACE_TIME_IDX,
    distance_good_key="eps_distance_sp_good",
    distance_bad_key="eps_distance_sp_bad",
    path_cost="max",
    score_label=r"$S_\infty(t,x)=C_{B,\infty}(t,x)-C_{G,\infty}(t,x)$",
)
surface_one = compute_time_layered_score_surface(
    adata,
    time_value=SURFACE_TIME_IDX,
    distance_good_key="eps_sum_distance_sp_good",
    distance_bad_key="eps_sum_distance_sp_bad",
    path_cost="sum",
    score_label=r"$S_1(t,x)=C_{B,1}(t,x)-C_{G,1}(t,x)$",
)

validate_background_queries(
    adata,
    time_value=SURFACE_TIME_IDX,
    output_key="eps_attracting_basin_sp",
    surface=surface_inf,
)
validate_background_queries(
    adata,
    time_value=SURFACE_TIME_IDX,
    output_key="eps_sum_attracting_basin_sp",
    surface=surface_one,
)

vabs_inf = _finite_symmetric_limit(surface_inf["score"])
vabs_one = _finite_symmetric_limit(surface_one["score"])
print(
    f"surface time_idx = {SURFACE_TIME_IDX}\n"
    f"S_inf range = [{np.nanmin(surface_inf['score']):.4f}, "
    f"{np.nanmax(surface_inf['score']):.4f}]\n"
    f"S_1 range   = [{np.nanmin(surface_one['score']):.4f}, "
    f"{np.nanmax(surface_one['score']):.4f}]"
)

# 1) 3D surfaces
fig = plt.figure(figsize=(15, 6))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
s1 = plot_score_surface_3d(ax1, surface_inf, vabs=vabs_inf, zlabel=r"$S_\infty$")
s2 = plot_score_surface_3d(ax2, surface_one, vabs=vabs_one, zlabel=r"$S_1$")
ax1.set_title(rf"$S_\infty$ at time_idx={SURFACE_TIME_IDX}")
ax2.set_title(rf"$S_1$ at time_idx={SURFACE_TIME_IDX}")
fig.colorbar(s1, ax=ax1, shrink=0.7, pad=0.08, label=r"$S_\infty$")
fig.colorbar(s2, ax=ax2, shrink=0.7, pad=0.08, label=r"$S_1$")
fig.tight_layout()
plt.show()

# 2) Publication-oriented 2D score maps
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.1), constrained_layout=True)
im1 = plot_score_map(
    axes[0],
    surface_inf,
    vabs=vabs_inf,
    title=rf"$S_\infty$ at time_idx={SURFACE_TIME_IDX}",
    show_legend=True,
)
im2 = plot_score_map(
    axes[1],
    surface_one,
    vabs=vabs_one,
    title=rf"$S_1$ at time_idx={SURFACE_TIME_IDX}",
)
fig.colorbar(im1, ax=axes[0], shrink=0.95, pad=0.02, label=r"$S_\infty$")
fig.colorbar(im2, ax=axes[1], shrink=0.95, pad=0.02, label=r"$S_1$")
fig.suptitle(r"Time-layered score maps; black curve: $S_p(t,x)=0$", y=1.02)
if SAVE_SCORE_SURFACE_FIGURES:
    fig.savefig(
        SCORE_SURFACE_OUTPUT_DIR / "score_surface_time_layered.png",
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        SCORE_SURFACE_OUTPUT_DIR / "score_surface_time_layered.pdf",
        bbox_inches="tight",
    )
plt.show()

# 3) Diagnostic decomposition for p=1: C_G, C_B, and S_1
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.4), constrained_layout=True)
cg = axes[0].contourf(surface_one["X"], surface_one["Y"], surface_one["cost_good"], levels=25)
cb = axes[1].contourf(surface_one["X"], surface_one["Y"], surface_one["cost_bad"], levels=25)
ss = plot_score_map(axes[2], surface_one, vabs=vabs_one, title=r"$S_1=C_{B,1}-C_{G,1}$")
axes[0].set_title(r"$C_{G,1}$")
axes[1].set_title(r"$C_{B,1}$")
for ax in axes[:2]:
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.colorbar(cg, ax=axes[0], shrink=0.9)
fig.colorbar(cb, ax=axes[1], shrink=0.9)
fig.colorbar(ss, ax=axes[2], shrink=0.9)
fig.suptitle(rf"Additive reach costs at time_idx={SURFACE_TIME_IDX}", y=1.02)
plt.show()

### Prediction accuracy along the trajectory

This shows when the eventual terminal class becomes distinguishable as the
held-out trajectory progresses.

In [ ]:
accuracy_by_step = (
    cv_eps.groupby("step", as_index=False)
    .agg(
        accuracy=("correct", "mean"),
        tie_rate=("is_tie", "mean"),
        mean_fraction=("fraction", "mean"),
    )
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    accuracy_by_step["step"],
    accuracy_by_step["accuracy"],
    "o-",
    label="accuracy",
)
ax.plot(
    accuracy_by_step["step"],
    accuracy_by_step["tie_rate"],
    "o--",
    label="exact-tie rate",
)
ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("step in held-out trajectory")
ax.set_ylabel("fraction")
ax.set_title("LOSO-CV performance along trajectory")
ax.grid(ls="--", alpha=0.4)
ax.legend()
plt.show()

accuracy_by_step


### $\varepsilon_G$ and $\varepsilon_B$ for representative held-out trajectories

In [ ]:
example_good_seq = terminal_cv.loc[
    terminal_cv["true_label"] == "good", "seq_id"
].iloc[0]
example_bad_seq = terminal_cv.loc[
    terminal_cv["true_label"] == "bad", "seq_id"
].iloc[0]

for sid in [example_good_seq, example_bad_seq]:
    df = cv_eps[cv_eps["seq_id"] == sid].sort_values("step")
    label = df["true_label"].iloc[0]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df["step"], df["epsilon_good"], "o-", label=r"$\varepsilon_G$")
    ax.plot(df["step"], df["epsilon_bad"], "o-", label=r"$\varepsilon_B$")
    ax.set_xlabel("step")
    ax.set_ylabel(r"minimum $\varepsilon$")
    ax.set_title(f"Held-out seq_id={sid} (true terminal: {label})")
    ax.grid(ls="--", alpha=0.4)
    ax.legend()
    plt.show()

### Cross-validated Good/Bad margin in state space

We define

$$
M(x)=\varepsilon_B(x)-\varepsilon_G(x).
$$

Positive values favor Good and negative values favor Bad.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    cv_eps["x"],
    cv_eps["y"],
    c=cv_eps["margin_good"],
    s=18,
    alpha=0.8,
)
ax.axhline(0.0, lw=0.8, alpha=0.4)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(r"LOSO-CV margin $\varepsilon_B-\varepsilon_G$")
ax.grid(ls="--", alpha=0.3)
plt.colorbar(sc, ax=ax, label=r"$\varepsilon_B-\varepsilon_G$")
plt.show()

### LOSO-CV for $\varepsilon_\Sigma$ ($p=1$)

Pointwise reliability for a new state is calibrated **only from out-of-sample
scores**. Therefore we also compute strict leave-one-sequence-out scores for
$p=1$ rather than calibrating on the in-sample landscape.


In [ ]:
cv_eps_sum = leave_one_sequence_out_cv(
    adata,
    path_cost="sum",
    seq_key="seq_id",
    time_key="time_idx",
    cluster_key="cluster",
    good_cluster_key="good",
    bad_cluster_key="bad",
    cost_matrix_key="cost_matrix",
    verbose=True,
)

display(cv_eps_sum.head(10))


## Pointwise prediction reliability for an arbitrary state $x$

A single state does not have an ordinary sample **accuracy**.  Instead, we
estimate the probability that the fixed landscape-sign prediction is correct.
For each time layer and each perturbation norm, strict LOSO scores are used to
fit the monotone calibration map

$$
q_{p,t}(s) \approx P(Y=G\mid S_p=s,t),
\qquad
S_p=C_{B,p}-C_{G,p}.
$$

The landscape prediction itself is kept fixed:

$$
\hat Y_p(x)=G \quad\text{if } S_p(x)>0,
\qquad
\hat Y_p(x)=B \quad\text{if } S_p(x)\le 0.
$$

For a new state $x$, the **pointwise prediction reliability** is

$$
R_p(x)=
\begin{cases}
q_{p,t}(S_p(x)), & \hat Y_p(x)=G,\\
1-q_{p,t}(S_p(x)), & \hat Y_p(x)=B.
\end{cases}
$$

Here we use time-specific isotonic regression.  This is intentionally a simple
calibration example: with only 40 trajectories, the fitted curve is a
staircase and should not be interpreted as a precise probability estimate.
For publication-level uncertainty quantification, nested/cross-fitted
calibration or a pooled time-dependent probabilistic model should be used.


In [ ]:
def fit_timewise_isotonic_calibrators(
    cv_frame,
    *,
    score_column="margin_good",
    time_column="time_idx",
    label_column="true_label",
    good_label="good",
):
    """Fit q_t(s) ~= P(Good | S=s, time=t) from OOS/LOSO scores only."""
    calibrators = {}
    summary_rows = []

    for time_value, group in cv_frame.groupby(time_column, sort=True):
        scores = group[score_column].to_numpy(dtype=float)
        y_good = (
            group[label_column].astype(str).to_numpy() == good_label
        ).astype(float)

        finite = np.isfinite(scores)
        scores = scores[finite]
        y_good = y_good[finite]
        if scores.size < 2:
            raise ValueError(
                f"Not enough finite calibration scores at time={time_value}."
            )

        model = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            increasing=True,
            out_of_bounds="clip",
        )
        model.fit(scores, y_good)

        calibrators[int(time_value)] = {
            "model": model,
            "score_min": float(np.min(scores)),
            "score_max": float(np.max(scores)),
            "n_samples": int(scores.size),
            "good_fraction": float(np.mean(y_good)),
        }
        summary_rows.append(
            {
                "time_idx": int(time_value),
                "n_samples": int(scores.size),
                "score_min": float(np.min(scores)),
                "score_max": float(np.max(scores)),
                "good_fraction": float(np.mean(y_good)),
            }
        )

    return calibrators, pd.DataFrame(summary_rows)


calibrators_inf, calibration_summary_inf = (
    fit_timewise_isotonic_calibrators(cv_eps)
)
calibrators_one, calibration_summary_one = (
    fit_timewise_isotonic_calibrators(cv_eps_sum)
)

print("L-infinity calibration summary")
display(calibration_summary_inf.head())
print("L1 calibration summary")
display(calibration_summary_one.head())


### Reusable evaluator: $C_G$, $C_B$, $S$, optimal paths, and reliability

The following function can be called for any external state `x`, provided that
`costs` contains the cost from that state to the ensemble observations.  In
time-layered mode, only the costs to the requested time layer need to be finite.


In [ ]:
def evaluate_query_with_reliability(
    adata_obj,
    x,
    costs,
    *,
    output_key,
    time_value,
    calibrators,
):
    """Evaluate an external x and attach calibrated pointwise reliability."""
    result = minimum_epsilon_and_paths_to_good_bad(
        adata_obj,
        x,
        costs,
        output_key=output_key,
        time_value=time_value,
    )

    score = float(result.epsilon_bad - result.epsilon_good)
    prediction = "good" if score > 0.0 else "bad"

    time_key = int(time_value)
    if time_key not in calibrators:
        raise KeyError(
            f"No calibration model is available for time={time_value}."
        )
    calibration = calibrators[time_key]
    probability_good = float(
        calibration["model"].predict([score])[0]
    )
    probability_bad = 1.0 - probability_good
    reliability = (
        probability_good
        if prediction == "good"
        else probability_bad
    )

    score_min = calibration["score_min"]
    score_max = calibration["score_max"]
    within_support = bool(score_min <= score <= score_max)

    return {
        "path_result": result,
        "C_G": float(result.epsilon_good),
        "C_B": float(result.epsilon_bad),
        "S": score,
        "prediction": prediction,
        "P_good": probability_good,
        "P_bad": probability_bad,
        "reliability": float(reliability),
        "within_calibration_support": within_support,
        "calibration_score_min": score_min,
        "calibration_score_max": score_max,
    }


### Evaluate the same external query $x$

The previously selected `x_query` is now assigned not only $C_G$, $C_B$, and
$S$, but also $P(G\mid S,t)$ and the estimated probability that the fixed sign
prediction is correct.


In [ ]:
query_reliability_inf = evaluate_query_with_reliability(
    adata,
    x_query,
    query_costs,
    output_key="eps_attracting_basin_sp",
    time_value=QUERY_TIME_IDX,
    calibrators=calibrators_inf,
)
query_reliability_one = evaluate_query_with_reliability(
    adata,
    x_query,
    query_costs,
    output_key="eps_sum_attracting_basin_sp",
    time_value=QUERY_TIME_IDX,
    calibrators=calibrators_one,
)

query_reliability_table = pd.DataFrame(
    [
        {
            "p": "infinity",
            **{
                key: query_reliability_inf[key]
                for key in (
                    "C_G",
                    "C_B",
                    "S",
                    "prediction",
                    "P_good",
                    "P_bad",
                    "reliability",
                    "within_calibration_support",
                    "calibration_score_min",
                    "calibration_score_max",
                )
            },
        },
        {
            "p": "1",
            **{
                key: query_reliability_one[key]
                for key in (
                    "C_G",
                    "C_B",
                    "S",
                    "prediction",
                    "P_good",
                    "P_bad",
                    "reliability",
                    "within_calibration_support",
                    "calibration_score_min",
                    "calibration_score_max",
                )
            },
        },
    ]
)

display(query_reliability_table)

for p_name, item in [
    ("L-infinity", query_reliability_inf),
    ("L1", query_reliability_one),
]:
    support_note = (
        "inside"
        if item["within_calibration_support"]
        else "OUTSIDE"
    )
    print(
        f"{p_name}: prediction={item['prediction']}, "
        f"estimated reliability={item['reliability']:.3f}, "
        f"P(G)={item['P_good']:.3f}, "
        f"score support={support_note} "
        f"[{item['calibration_score_min']:.3f}, "
        f"{item['calibration_score_max']:.3f}]"
    )


### Calibration curves at the query time

Dots are the strict LOSO observations at the same time layer.  The solid curve
is the isotonic estimate of $P(G\mid S_p,t)$, and the star is the external
query state.  The vertical dashed line at $S=0$ is the fixed landscape decision
boundary; it is **not** refitted during calibration.


In [ ]:
def plot_query_calibration(
    ax,
    *,
    cv_frame,
    calibrators,
    query_info,
    time_value,
    title,
):
    group = cv_frame.loc[
        cv_frame["time_idx"].to_numpy(dtype=int) == int(time_value)
    ].copy()
    scores = group["margin_good"].to_numpy(dtype=float)
    y_good = (
        group["true_label"].astype(str).to_numpy() == "good"
    ).astype(float)

    calibration = calibrators[int(time_value)]
    s_min = calibration["score_min"]
    s_max = calibration["score_max"]
    if np.isclose(s_min, s_max):
        pad = max(1e-6, 0.1 * max(1.0, abs(s_min)))
        grid = np.linspace(s_min - pad, s_max + pad, 200)
    else:
        pad = 0.08 * (s_max - s_min)
        grid = np.linspace(s_min - pad, s_max + pad, 300)

    p_good_grid = calibration["model"].predict(grid)

    rng_local = np.random.default_rng(0)
    jitter = rng_local.normal(0.0, 0.018, size=len(y_good))
    ax.scatter(
        scores,
        np.clip(y_good + jitter, -0.06, 1.06),
        s=24,
        alpha=0.55,
        label="LOSO outcomes",
    )
    ax.plot(
        grid,
        p_good_grid,
        linewidth=2.2,
        label=r"isotonic $P(G\mid S,t)$",
    )
    ax.axvline(0.0, color="0.4", ls="--", lw=1.0)
    ax.axhline(0.5, color="0.6", ls=":", lw=1.0)
    ax.scatter(
        [query_info["S"]],
        [query_info["P_good"]],
        marker="*",
        s=180,
        color="black",
        edgecolor="white",
        linewidth=0.8,
        zorder=10,
        label="query x",
    )

    ax.set_ylim(-0.08, 1.08)
    ax.set_xlabel(r"landscape score $S_p=C_{B,p}-C_{G,p}$")
    ax.set_ylabel(r"estimated $P(G\mid S_p,t)$")
    ax.set_title(
        title
        + "\n"
        + rf"$C_G={query_info['C_G']:.3f}$, "
          rf"$C_B={query_info['C_B']:.3f}$, "
          rf"$S={query_info['S']:.3f}$, "
          rf"$R={query_info['reliability']:.2f}$"
    )
    ax.grid(ls="--", alpha=0.25)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.8),
    constrained_layout=True,
)
plot_query_calibration(
    axes[0],
    cv_frame=cv_eps,
    calibrators=calibrators_inf,
    query_info=query_reliability_inf,
    time_value=QUERY_TIME_IDX,
    title=rf"$p=\infty$, time_idx={QUERY_TIME_IDX}",
)
plot_query_calibration(
    axes[1],
    cv_frame=cv_eps_sum,
    calibrators=calibrators_one,
    query_info=query_reliability_one,
    time_value=QUERY_TIME_IDX,
    title=rf"$p=1$, time_idx={QUERY_TIME_IDX}",
)
axes[0].legend(frameon=True, fontsize=8, loc="best")
fig.suptitle(
    "Pointwise prediction reliability calibrated from strict LOSO scores",
    y=1.02,
)
plt.show()


### Global accuracy versus pointwise reliability

`accuracy(t)` below is a property of the collection of held-out trajectories,
whereas `reliability(x)` is a property assigned to the single external query
through the calibrated LOSO score distribution.  They should not be called the
same quantity.


In [ ]:
def accuracy_by_time(frame):
    return (
        frame.groupby("time_idx", as_index=False)
        .agg(
            accuracy=("correct", "mean"),
            n=("correct", "size"),
        )
    )

accuracy_inf_time = accuracy_by_time(cv_eps)
accuracy_one_time = accuracy_by_time(cv_eps_sum)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(
    accuracy_inf_time["time_idx"],
    accuracy_inf_time["accuracy"],
    "o-",
    label=r"LOSO accuracy, $p=\infty$",
)
ax.plot(
    accuracy_one_time["time_idx"],
    accuracy_one_time["accuracy"],
    "s-",
    label=r"LOSO accuracy, $p=1$",
)

ax.scatter(
    [QUERY_TIME_IDX],
    [query_reliability_inf["reliability"]],
    marker="*",
    s=170,
    label=r"query reliability, $p=\infty$",
)
ax.scatter(
    [QUERY_TIME_IDX],
    [query_reliability_one["reliability"]],
    marker="X",
    s=90,
    label=r"query reliability, $p=1$",
)

ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("time_idx")
ax.set_ylabel("fraction / probability")
ax.set_title(
    "Global LOSO accuracy and local calibrated reliability of the query"
)
ax.grid(ls="--", alpha=0.3)
ax.legend(frameon=True, fontsize=8)
plt.show()


### Interpretation for control-oriented use

For an arbitrary state $x$, the notebook now returns two different kinds of
information:

1. **Predictability:** the fixed sign prediction and its calibrated reliability
   $R_p(x)$.
2. **Accessibility/control cost:** $C_{G,p}(x)$, $C_{B,p}(x)$, and the optimal
   paths realizing those costs.

For example, if the state is predicted Bad with high $R_p(x)$ but
$C_{G,p}(x)$ remains below an admissible perturbation budget, the state is both
sufficiently diagnosed and still redirectable toward Good under the assumed
controlled-path model.  This is the intended bridge between prediction and
candidate intervention-path analysis.
